# STUDI KASUS: NAVIGASI ROBOT PEMBERSIH (GRID WORLD)

## 1. LATAR BELAKANG DAN MASALAH

Sebuah robot pembersih berada di dalam ruangan berukuran **3 x 3**.

Robot bertugas mencari sampah untuk dibersihkan sambil menghindari:

* Area basah atau berbahaya
* Stasiun pengisian listrik terbuka
* Bahaya air
* Dinding

### Tujuan

Robot belajar mencari rute tercepat dan teraman dari titik awal menuju lokasi sampah secara otomatis, tanpa diprogram rutenya secara manual.

---

## 2. ELEMEN UTAMA REINFORCEMENT LEARNING (RL)

### State (s)

State merupakan posisi robot pada grid `(x, y)`.

Terdapat total **9 petak**, yaitu:

* Grid 0
* Grid 1
* Grid 2
* Grid 3
* Grid 4
* Grid 5
* Grid 6
* Grid 7
* Grid 8

### Action (a)

Robot memiliki 4 pilihan arah gerakan:

* Atas
* Bawah
* Kiri
* Kanan

### Reward (r)

|   Reward | Kondisi                                                       |
| -------: | ------------------------------------------------------------- |
| **+100** | Berhasil mencapai petak sampah (Goal)                         |
|  **-50** | Jatuh atau masuk ke petak bahaya (Hazard)                     |
|   **-1** | Setiap langkah biasa. Mendorong robot mencari jalur terpendek |
|  **-10** | Menabrak dinding. Robot tetap berada di posisi semula         |

---

## 3. LAYOUT PETA

Ukuran grid: **3 x 3**

<table>
  <tr>
    <td align="center"><b>[0] START ROBOT</b><br>(0,0)</td>
    <td align="center"><b>[1]</b><br>(0,1)</td>
    <td align="center"><b>[2] BAHAYA</b><br>(0,2)</td>
  </tr>
  <tr>
    <td align="center"><b>[3]</b><br>(1,0)</td>
    <td align="center"><b>[4] DINDING</b><br>(1,1)</td>
    <td align="center"><b>[5]</b><br>(1,2)</td>
  </tr>
  <tr>
    <td align="center"><b>[6]</b><br>(2,0)</td>
    <td align="center"><b>[7]</b><br>(2,1)</td>
    <td align="center"><b>[8] SAMPAH</b><br>(2,2)</td>
  </tr>
</table>

### Keterangan

* **Grid 0** : Titik awal robot (START)
* **Grid 4** : Dinding
* **Grid 2** : Area bahaya (HAZARD)
* **Grid 8** : Lokasi sampah (GOAL)

---

## 4. Q-TABLE

Pada awal latihan (**epoch 0**), Q-Table diinisialisasi dengan ukuran:

**9 x 4**

Artinya:

* **9 state/grid**
* **4 action/arah gerakan**

Semua nilai Q pada awal latihan diinisialisasi dengan nilai:

**0.4**

---

## 5. ALUR PELATIHAN (TRAINING LOOP)

Robot dilatih selama beberapa episode menggunakan algoritma **Q-Learning**.

### Rumus Pembaruan Q-Learning

$$
Q(s, a) = Q(s, a) + \alpha \left[r + \gamma \max_{a'} Q(s', a') - Q(s, a)\right]
$$

---

## 6. HYPERPARAMETER

### Learning Rate (α)

**0.1**

Learning rate menentukan seberapa besar nilai Q lama diperbarui berdasarkan pengalaman baru.

### Discount Factor (γ)

**0.9**

Discount factor menentukan seberapa besar robot mempertimbangkan reward jangka panjang.

### Epsilon (ε)

**0.2**

Epsilon digunakan dalam strategi **epsilon-greedy** untuk menentukan tingkat eksplorasi.

Dengan ε = 0.2:

* **20%** kemungkinan robot melakukan eksplorasi atau mencoba aksi secara acak.
* **80%** kemungkinan robot memilih aksi dengan nilai Q tertinggi.

---

## 7. PROSES BELAJAR

### EPISODE AWAL

Pada awal latihan, robot masih banyak bergerak secara acak.

Robot dapat:

* Menabrak dinding dan mendapatkan reward **-10**.
* Masuk ke area bahaya dan mendapatkan reward **-50**.
* Bergerak menuju petak lain tanpa mengetahui jalur terbaik.

Robot belum mengetahui jalur menuju sampah.

### EPISODE MENENGAH

Setelah mendapatkan pengalaman dari beberapa episode, robot mulai belajar.

Ketika robot berhasil mencapai petak sampah, robot mendapatkan reward **+100**.

Nilai reward tersebut kemudian memengaruhi nilai Q pada state dan action yang mengarah menuju goal.

Reward dari petak sampah secara bertahap **"menjalar"** ke state-state sebelumnya melalui proses pembaruan Q-Learning.

Akibatnya, robot mulai mengetahui jalur mana yang lebih menguntungkan.

### EPISODE AKHIR

Setelah melalui banyak episode, nilai Q-Table semakin stabil dan mendekati konvergensi.

Robot mulai mengetahui:

* Aksi yang mengarah menuju sampah.
* Aksi yang harus dihindari karena menuju bahaya.
* Aksi yang menyebabkan robot menabrak dinding.
* Jalur yang lebih pendek menuju goal.

Pada tahap ini, robot cenderung memilih aksi dengan nilai Q tertinggi pada setiap state.

Dengan demikian, robot dapat menemukan rute yang lebih aman dan lebih cepat menuju lokasi sampah tanpa rute tersebut diprogram secara manual.

---

## 8. INTI DARI STUDI KASUS

Robot menggunakan pengalaman untuk belajar.

### Pada awalnya

Robot tidak mengetahui jalur terbaik.

### Selama training

Robot mencoba berbagai aksi dan menerima reward atau punishment.

### Setelah banyak episode

Nilai Q-Table diperbarui berdasarkan pengalaman.

### Pada akhirnya

Robot memilih aksi dengan nilai Q terbaik sehingga dapat menemukan jalur yang aman dan efisien menuju sampah.


In [1]:
import numpy as np

# Inisialisasi Environment (3x3 grid)
# State: 0..8, Action: 0=Up, 1=Down, 2=Left, 3=Right
num_states = 9
num_actions = 4
q_table = np.zeros((num_states, num_actions))

alpha = 0.1
gamma = 0.9
epsilon = 0.2

# Posisi khusus
START = 0
HAZARD = 2
WALL = 4
GOAL = 8

def get_next_state_and_reward(state, action):
    # Mapping pergerakan grid 3x3
    row, col = divmod(state, 3)

    if action == 0 and row > 0: row -= 1    # Up
    elif action == 1 and row < 2: row += 1  # Down
    elif action == 2 and col > 0: col -= 1  # Left
    elif action == 3 and col < 2: col += 1  # Right

    next_state = row * 3 + col

    # Penanganan dinding & reward
    if next_state == WALL:
        return state, -10, False  # Nabrak dinding, posisi tidak berubah
    elif next_state == HAZARD:
        return next_state, -50, True  # Masuk bahaya, episode selesai
    elif next_state == GOAL:
        return next_state, 100, True  # Berhasil, episode selesai
    else:
        return next_state, -1, False  # Langkah biasa

# Training loop
for episode in range(1000):
    state = START
    done = False

    while not done:
        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.randint(num_actions)
        else:
            action = np.argmax(q_table[state])

        next_state, reward, done = get_next_state_and_reward(state, action)

        # Q-learning update formula
        best_next_action = np.argmax(q_table[next_state])
        q_table[state, action] += alpha * (
            reward + gamma * q_table[next_state, best_next_action] - q_table[state, action]
        )

        state = next_state

print("Hasil Q-Table setelah training:")
print(np.round(q_table, 1))

Hasil Q-Table setelah training:
[[ 61.7  70.2  62.   53.8]
 [ 10.6   4.6  62.  -20.5]
 [  0.    0.    0.    0. ]
 [ 62.   79.1  69.9  61. ]
 [  0.    0.    0.    0. ]
 [  0.    0.    0.    0. ]
 [ 70.   78.8  79.1  89. ]
 [ 79.9  88.9  78.6 100. ]
 [  0.    0.    0.    0. ]]
